In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
model_name = "scb10x/llama3.2-typhoon2-1b-instruct"

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

[2025-05-15 09:44:54,014] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to mps (auto detect)


W0515 09:44:54.178000 34238 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital of France is Paris.<|eot_id|>


In [6]:
# Import dataset
import json
import os

def get_data_from_json(file_path):
    """Load data from a JSON file"""
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

base_path = os.getcwd()
# Load data from JSON file
training_data = get_data_from_json(os.path.join(base_path, "data/klon8_data/non_instruct_tagged_train_pretrain.json"))
new_token = get_data_from_json(os.path.join(base_path, "data/klon8_data/phonetic_token.json"))


In [3]:
tmp_data_1 = get_data_from_json(os.path.join(base_path, "data/klon8_data/instruct_untagged_train_sft.json"))
tmp_data_2 = get_data_from_json(os.path.join(base_path, "data/klon8_data/instruct_phonetic_nmt_train_sft.json"))

len(tmp_data_1) + len(tmp_data_2)

39943

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Pongsaky/llama3.2-typhoon2-1b-instruct-klon8-llama_factory")
tokenizer.tokenize("<r>[a][w]</r>")

tokenizer_config.json:   0%|          | 0.00/66.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

['<r>', '[a]', '[w]', '</r>']

In [14]:
def convert_sharegpt_to_openai_format(data: dict, message_key: str, from_key: str, content_key: str, user_role: str, assistant_role: str, system_key: str = None):
    """
    Convert ShareGPT format to OpenAI format
    """
    if message_key not in data.keys():
        raise ValueError(f"message_key {message_key} not found in data")
    
    messages = data[message_key]
    openai_messages = []
    
    if system_key is not None:
        if system_key not in data.keys():
            raise ValueError(f"system_key {system_key} not found in data")
        openai_messages.append({"role": "system", "content": data[system_key]})
    
    for item in messages:
        if item[from_key] not in [user_role, assistant_role]:
            raise ValueError(f"from_key {item[from_key]} not in [user_role, assistant_role]")
        
        role = "user" if item[from_key] == user_role else "assistant"
        openai_messages.append({"role": role, "content": item[content_key]})
        
    return openai_messages


In [19]:
openai_message = convert_sharegpt_to_openai_format(tmp_data_2[0], message_key="conversations", from_key="from", content_key="content", user_role="human", assistant_role="model", system_key="system")
tokenizer.apply_chat_template(openai_message, add_generation_prompt=True, padding=True, return_tensors="pt", tokenize=False)

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are an AI specialized in Thai phonetic transcription. When given a Thai sentence in Thai script, your task is to output its phonetic transcription using the following format. Each syllable must be represented in a phoneme block as [StarterConsonant][Vowel][EndingConsonant][ToneNumber], omitting the EndingConsonant if it does not exist. Use the tilde symbol `~` to link syllables that are part of the same word, and use the vertical bar `|` to separate different words. For any short syllable, append a single apostrophe `'` immediately after its phoneme block. Do not include any glottal stop or aspiration markers in the transcription. Your output must contain only the transcription with no explanation, translation, or additional text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nเข้ากันไม่ค่อยดี<|eot_id|><|start_header_id|>assistant<|end_header_id|>\

In [20]:
tmp_data_2[0]

{'conversations': [{'from': 'human', 'content': 'เข้ากันไม่ค่อยดี'},
  {'from': 'model',
   'content': '[kh][a][w][2]|[k][a][n][0]|[m][a][j][2]~[kh][OO][j][2]|[d][ii][0]|'}],
 'system': "You are an AI specialized in Thai phonetic transcription. When given a Thai sentence in Thai script, your task is to output its phonetic transcription using the following format. Each syllable must be represented in a phoneme block as [StarterConsonant][Vowel][EndingConsonant][ToneNumber], omitting the EndingConsonant if it does not exist. Use the tilde symbol `~` to link syllables that are part of the same word, and use the vertical bar `|` to separate different words. For any short syllable, append a single apostrophe `'` immediately after its phoneme block. Do not include any glottal stop or aspiration markers in the transcription. Your output must contain only the transcription with no explanation, translation, or additional text."}

In [24]:
max_token_data = -1e9
min_token_data = 1e9

for data in tmp_data_2:
    openai_message = convert_sharegpt_to_openai_format(data, message_key="conversations", from_key="from", content_key="content", user_role="human", assistant_role="model", system_key="system")
    input_ids = tokenizer.apply_chat_template(openai_message, add_generation_prompt=True, padding=True, return_tensors="pt")
    max_token_data = max(max_token_data, input_ids.shape[1])
    min_token_data = min(min_token_data, input_ids.shape[1])
print("Max token data: ", max_token_data)
print("Min token data: ", min_token_data)

Max token data:  1749
Min token data:  215


In [26]:
import numpy as np

# Calculate steps
total_train_samples = 20000
# max_samples = 1000
# total_train_samples = min(total_train_samples, max_samples)
batch_size = 64
accumulation_steps = 2
epochs = 2

save_steps_per_epoch = 1
eval_times_per_epoch = 2
every_n_steps_per_epoch = 30

total_steps = int(np.ceil(total_train_samples / (batch_size * accumulation_steps)) * epochs)
every_n_steps = total_steps // (epochs * every_n_steps_per_epoch)
warmup_steps = int(0.1 * total_steps) # 10% warmup
# Evaluate and save based on save_steps_per_epoch
save_steps = int(np.ceil(total_steps / (save_steps_per_epoch * epochs)))
eval_steps = total_steps // (epochs * eval_times_per_epoch)

# # Calculate eval_steps to ensure it divides save_steps evenly
# ideal_eval_frequency = total_steps // (epochs * eval_times_per_epoch)
# # Find the largest divisor of save_steps that is <= ideal_eval_frequency
# for divisor in range(ideal_eval_frequency, 0, -1):
#     if save_steps % divisor == 0:
#         eval_steps = divisor
#         break

# Calculated Steps Summary
print("  Calculated Steps:")
print(f"    Total Steps: {total_steps}")
print(f"    Warmup Steps: {warmup_steps}")
print(f"    Save Steps: {save_steps}")
print(f"    Eval Steps: {eval_steps}")
print(f"    Every N Steps: {every_n_steps}")
print("-" * 50)

  Calculated Steps:
    Total Steps: 314
    Warmup Steps: 31
    Save Steps: 157
    Eval Steps: 78
    Every N Steps: 5
--------------------------------------------------


In [10]:
save_steps % eval_steps

0

In [ ]:
# wandb sync /workspace/LLaMA-Factory-klon8-generation/wandb/offline-run-20250515_175946-dg2bjwyp

In [92]:
total_steps/eval_steps

14.0

In [4]:
new_token[:5], training_data[0]

(['[0]', '[1]', '[2]', '[3]', '[4]'],
 {'text': '<klon8> พอได้ยินเสียงระฆังข้างหลัง<r>[a][w]เขา</r>\tเห็นผู้<r>[a][w]เฒ่า</r>ออกจากชะวาก<r>[a]ผา</r>\nดูสรรพางค์ร่างกายแก่ช<r>[a]รา</r>\tแต่ผิว<r>[a]หน้า</r>นั้นละม้ายคล้ายทา<r>[o][k]รก</r>\nทรงเสื้อโขมพัสตรานุ่งผ้า<r>[a][w]ขาว</r>\tผมนั้น<r>[a][w]ยาว</r>ย้อยสยายประปราย<r>[o][k]ปรก</r>\nถือไม้เท้าเนาวรัตน์พัดขน<r>[o][k]นก</r>\tทำเดิน<r>[o][k]งก</r>งันมาแล้วพา<r>[i]ที</r>\nว่าดูราสามีนางผี<r>[Ua]เสื้อ</r>\tเป็นหน่อ<r>[Ua]เนื้อ</r>กษัตริย์ชาติราช<r>[i]สีห์</r>\nอย่าเผาศพนางยักษ์ด้วยอัค<r>[i]คี</r>\tภัยจะ<r>[i]มี</r>ถึงกายให้วาย<r>[a][n]ปราณ</r>\n'})

Add new token to tokenizer and model with 2 apporaches
1. Mean initialize the weights of the new token
2. Prior knowlege from training data


In [5]:
old_tokenizer_vocab_size = len(tokenizer)
old_model_vocab_size = model.vocab_size
old_tokenizer_vocab_size, old_model_vocab_size

(128256, 128256)

In [6]:
# Approach 1: Mean initialize the weights of the new token

tokenizer.add_tokens(new_token)
model.resize_token_embeddings(len(tokenizer))

new_tokenizer_vocab_size = len(tokenizer)
new_model_vocab_size = model.vocab_size

new_tokenizer_vocab_size, new_model_vocab_size

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


(128334, 128334)

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital of France is Paris.<|eot_id|>


In [10]:
model.save_pretrained("llama3.2-typhoon2-1b-instruct-new-token")
tokenizer.save_pretrained("llama3.2-typhoon2-1b-instruct-new-token")

('llama3.2-typhoon2-1b-instruct-new-token/tokenizer_config.json',
 'llama3.2-typhoon2-1b-instruct-new-token/special_tokens_map.json',
 'llama3.2-typhoon2-1b-instruct-new-token/tokenizer.json')

In [3]:
model = AutoModelForCausalLM.from_pretrained("llama3.2-typhoon2-1b-instruct-new-token")
tokenizer = AutoTokenizer.from_pretrained("llama3.2-typhoon2-1b-instruct-new-token")

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))


[2025-05-15 11:04:54,085] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to mps (auto detect)


W0515 11:04:54.266000 35433 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital of France is Paris.<|eot_id|>


In [4]:
model.vocab_size

128334

In [14]:
training_data[0]

{'text': '<klon8> พอได้ยินเสียงระฆังข้างหลัง<r>[a][w]เขา</r>\tเห็นผู้<r>[a][w]เฒ่า</r>ออกจากชะวาก<r>[a]ผา</r>\nดูสรรพางค์ร่างกายแก่ช<r>[a]รา</r>\tแต่ผิว<r>[a]หน้า</r>นั้นละม้ายคล้ายทา<r>[o][k]รก</r>\nทรงเสื้อโขมพัสตรานุ่งผ้า<r>[a][w]ขาว</r>\tผมนั้น<r>[a][w]ยาว</r>ย้อยสยายประปราย<r>[o][k]ปรก</r>\nถือไม้เท้าเนาวรัตน์พัดขน<r>[o][k]นก</r>\tทำเดิน<r>[o][k]งก</r>งันมาแล้วพา<r>[i]ที</r>\nว่าดูราสามีนางผี<r>[Ua]เสื้อ</r>\tเป็นหน่อ<r>[Ua]เนื้อ</r>กษัตริย์ชาติราช<r>[i]สีห์</r>\nอย่าเผาศพนางยักษ์ด้วยอัค<r>[i]คี</r>\tภัยจะ<r>[i]มี</r>ถึงกายให้วาย<r>[a][n]ปราณ</r>\n'}

In [15]:
import torch
import gc
from collections import defaultdict
import re

def extract_phonetic_combinations(training_data, tokenizer, model_type="gemma|llama"):
    assert(model_type in ["gemma", "llama"])
    # Define regex pattern to match <r>[X][Y]text</r> patterns
    pattern = r'<r>(.+?)</r>'

    tag_dict = defaultdict(set)

    # Process each text in the training data
    for data in training_data:
        # Find all matches of the pattern in the text
        text = data["text"]
        matches = re.findall(pattern, text)

        for item in matches:
            # Extract all tags (e.g., [a], [w])
            tags = re.findall(r'\[.*?\]', item)
            # Extract the word by removing tags
            word = re.sub(r'\[.*?\]', '', item).strip()
            # Add the word to each tag's set
            for tag in tags:
                if "<r>" in word:
                    print(
                        f"Warning: <r> tag found in word '{word}'. Skipping this entry.")
                    continue
                tag_dict[tag].add(word)

    new_tag_dict = defaultdict(set)
    for key, value in tag_dict.items():
        for word in value:
            if model_type == "gemma":
                tokenized_words = tokenizer.tokenizer.encode(word, add_special_tokens=False)
            else:
                tokenized_words = tokenizer.encode(word, add_special_tokens=False)
            # tokenized_words = tokenizer.encode(word, add_special_tokens=False)
            if "<r>" in tokenized_words:
                print(f"Value: {value}")
                print(f"Tokenized word: {word}")
                print(f"Tokenized word: {tokenized_words}")
            for tokenized_word in tokenized_words:
                new_tag_dict[key].add(tokenized_word)

    return new_tag_dict

# Custom function for Gemma-3 token adding
def mean_surround_new_tag_token(model, tag_dict):
    # Calculate the mean of surrounding token embeddings
    embedding_matrix = model.get_input_embeddings().weight.clone()
    lm_head_matrix = model.get_output_embeddings().weight.clone()

    tag_embedding_dict = {}
    tag_lm_head_dict = {}
    
    tag_embedding = torch.zeros_like(embedding_matrix[0])
    tag_lm_head = torch.zeros_like(lm_head_matrix[0])

    for tag, ids_values in tag_dict.items():
        # properly accumulate all the embeddings
        tag_embedding.zero_()
        tag_lm_head.zero_()
        for idx in ids_values:
            tag_embedding += embedding_matrix[idx]
            tag_lm_head += lm_head_matrix[idx]
        tag_embedding_dict[tag] = tag_embedding / len(ids_values)
        tag_lm_head_dict[tag] = tag_lm_head / len(ids_values)

    return tag_embedding_dict, tag_lm_head_dict


def add_new_tokens(
    model,
    tokenizer,
    tag_dict,
    new_tokens=[],
    model_type="gemma|llama"
):
    """
    Smartly resizes the tokenizer and adds new tokens to the model.
    We also disregard untrained tokens by removing them from the mean calculation.
    """
    # All Unsloth Zoo code licensed under LGPLv3
    assert (isinstance(new_tokens, (list, tuple)))
    assert (len(new_tokens) > 0)
    assert (len(tag_dict) > 0)
    assert (isinstance(tag_dict, dict))
    assert(model_type in ["gemma", "llama"])

    # Check if tokens already exist
    if model_type == "gemma":
        overlapping_tokens = set(new_tokens) & set(tokenizer.tokenizer.vocab.keys())
    else:
        overlapping_tokens = set(new_tokens) & set(tokenizer.vocab.keys())
        
    if len(overlapping_tokens) = 0:
        print(
            f"Unsloth: You're adding new_tokens = {new_tokens}\n"
            f"There are tokens which are overlapping = {list(overlapping_tokens)}\n"
            f"We shall safely ignore these overlapping tokens."
        )
        new_tokens = [x for x in new_tokens if x not in overlapping_tokens]


    # Weirdly be careful reserved tokens can pop out
    tag_embedding_dict, tag_lm_head_dict = mean_surround_new_tag_token(
        model, tag_dict)

    # Get old lengths
    old_input_embedding = model.get_input_embeddings().weight
    old_output_embedding = model.get_output_embeddings().weight
    old_input_length = old_input_embedding.shape[0]
    old_output_length = old_output_embedding.shape[0]
    if model_type == "gemma":
        old_config_size = model.config.text_config.vocab_size
    else:
        old_config_size = model.config.vocab_size

    # Check for tied weights as well
    is_tied = (old_input_embedding.data_ptr() == old_output_embedding.data_ptr()) \
        or (model.config.tie_word_embeddings)

    # Add tokens
    if model_type == "gemma":
        old_length = len(tokenizer.tokenizer)
        tokenizer.tokenizer.add_tokens(new_tokens)
        model.resize_token_embeddings(len(tokenizer.tokenizer))
    else:
        old_length = len(tokenizer)
        tokenizer.add_tokens(new_tokens)
        model.resize_token_embeddings(len(tokenizer))
    # Also resizes lm_head as well

    # the Word2Vec sum of the other vectors
    embedding_matrix = model.get_input_embeddings().weight
    lm_head_matrix = model.get_output_embeddings().weight

    # Confirm sizes are correct
    if embedding_matrix.shape[0] > (old_input_length + len(new_tokens)):
        raise RuntimeError(
            "Unsloth: Embedding matrix size did not get resized properly. Please file a bug report"
        )
    if lm_head_matrix.shape[0] > (old_output_length + len(new_tokens)):
        raise RuntimeError(
            "Unsloth: LM Head matrix size did not get resized properly. Please file a bug report"
        )
    if model_type == "gemma":
        if model.config.text_config.vocab_size > (old_config_size + len(new_tokens)):
            raise RuntimeError(
                "Unsloth: Model's config vocab_size did not get resized properly. Please file a bug report"
            )
    else:
        if model.config.vocab_size > (old_config_size + len(new_tokens)):
            raise RuntimeError(
                "Unsloth: Model's config vocab_size did not get resized properly. Please file a bug report"
            )

    key_list = list(tag_dict.keys())
    if model_type == "gemma":
        key_ids_list = [tokenizer.tokenizer.encode(word, add_special_tokens=False)[0] for word in key_list]
    else:
        key_ids_list = [tokenizer.encode(word, add_special_tokens=False)[0] for word in key_list]
    with torch.no_grad():
        for key, ids in zip(key_list, key_ids_list):
            tag_embedding = tag_embedding_dict[key]
            tag_lm_head = tag_lm_head_dict[key]
            embedding_matrix[ids] = tag_embedding
            lm_head_matrix[ids] = tag_lm_head
            pass

    # We set a flag to say we need to train embeddings
    internal_model = model
    while hasattr(internal_model, "model"):
        internal_model._need_to_train_embeddings = True
        internal_model = internal_model.model
    pass
    internal_model._need_to_train_embeddings = True
    
    # Fix up all vocab sizes
    current_model = model
    if hasattr(current_model, "model") and hasattr(current_model, "config"):
        if model_type == "gemma":
            if hasattr(current_model.config.text_config, "vocab_size"):
                current_model.config.text_config.update({"vocab_size": len(tokenizer.tokenizer)})
        else:
            if hasattr(current_model.config, "vocab_size"):
                current_model.config.update({"vocab_size": len(tokenizer)})
        current_model = current_model.model

    # Must tie lm_head and embed_tokens if they are tied
    # Otherwise error will occur on saving models ie use save_model
    if is_tied:
        model.tie_weights()

    # Clear deleted GPU items
    for _ in range(3):
        gc.collect()
        torch.cuda.empty_cache()
    return

In [16]:
tag_dict = extract_phonetic_combinations(training_data, tokenizer, model_type="llama")

In [23]:
add_new_tokens(model=model, tokenizer=tokenizer, tag_dict=tag_dict, new_tokens=new_token, model_type="llama")

Unsloth: You're adding new_tokens = ['[0]', '[1]', '[2]', '[3]', '[4]', '[?]', '[@@-]', '[@@]', '[@]', '[N]', '[OO-]', '[OO]', '[O]', '[UU]', '[UUa]', '[Ua]', '[U]', '[a-]', '[a]', '[aa]', '[aee]', '[b]', '[bl]', '[br]', '[c]', '[ch]', '[d]', '[dr]', '[e]', '[ee]', '[f]', '[fl]', '[fr]', '[h]', '[i]', '[ia]', '[ii-]', '[ii]', '[iia]', '[j]', '[k]', '[kh]', '[khl]', '[khr]', '[khw]', '[kl]', '[kr]', '[kw]', '[l]', '[m]', '[n]', '[o]', '[oo]', '[p]', '[ph]', '[phl]', '[phr]', '[pl]', '[pr]', '[r]', '[s]', '[sw]', '[t]', '[th]', '[thr]', '[tr]', '[u]', '[ua]', '[uu-]', '[uu]', '[uua]', '[w]', '[x]', '[xx-]', '[xx]', '<klon8>', '<r>', '</r>']
There are tokens which are overlapping = ['[oo]', '[ch]', '[kr]', '[khw]', '[aee]', '[k]', '[U]', '[d]', '[f]', '[ee]', '[uu-]', '[n]', '[o]', '[s]', '[a]', '[2]', '[xx-]', '[uu]', '[h]', '[phr]', '[kw]', '</r>', '<r>', '[3]', '[w]', '[p]', '[c]', '[b]', '[i]', '[th]', '[a-]', '[m]', '[iia]', '[r]', '[UUa]', '[j]', '[uua]', '[e]', '[l]', '[dr]', '[@@]

In [24]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("llama3.2-typhoon2-1b-smart-tokens")
tokenizer = AutoTokenizer.from_pretrained("llama3.2-typhoon2-1b-smart-tokens")

In [25]:
model.vocab_size


128334

In [2]:
import torch
device = "mps" if torch.mps.is_available() else "cpu"
device

'mps'

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "saves/llama3.2-typhoon2-1b-instruct-lora-klon8-nmt"
# model_id = "Pongsaky/llama3.2-typhoon2-1b-instruct-tagged_nmt-mixed"
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model.vocab_size, len(tokenizer)

/opt/homebrew/Caskroom/miniconda/base/envs/llama-factory/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:550: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


(128334, 128334)

In [5]:
model.push_to_hub("Pongsaky/llama3.2-typhoon2-1b-instruct-lora-klon8-nmt", safe_serialization=False)
tokenizer.push_to_hub("Pongsaky/llama3.2-typhoon2-1b-instruct-lora-klon8-nmt", safe_serialization=False)

/opt/homebrew/Caskroom/miniconda/base/envs/llama-factory/lib/python3.10/site-packages/peft/utils/save_and_load.py:220: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


adapter_model.bin:   0%|          | 0.00/1.30G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Pongsaky/llama3.2-typhoon2-1b-instruct-lora-klon8-nmt/commit/e6e4691a4a464457bc9707d2e8435f4ab9fe95b5', commit_message='Upload tokenizer', commit_description='', oid='e6e4691a4a464457bc9707d2e8435f4ab9fe95b5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Pongsaky/llama3.2-typhoon2-1b-instruct-lora-klon8-nmt', endpoint='https://huggingface.co', repo_type='model', repo_id='Pongsaky/llama3.2-typhoon2-1b-instruct-lora-klon8-nmt'), pr_revision=None, pr_num=None)

In [9]:
messages = [
    {"role": "system", "content": "You are an AI specialized in Thai phonetic transcription. When given a Thai sentence in Thai script, your task is to output its phonetic transcription using the following format. Each syllable must be represented in a phoneme block as [StarterConsonant][Vowel][EndingConsonant][ToneNumber], omitting the EndingConsonant if it does not exist. Use the tilde symbol `~` to link syllables that are part of the same word, and use the vertical bar `|` to separate different words. For any short syllable, append a single apostrophe `'` immediately after its phoneme block. Do not include any glottal stop or aspiration markers in the transcription. Your output must contain only the transcription with no explanation, translation, or additional text."},
    {"role": "user", "content": "และมันไม่ได้ดูดีอย่างที่เห็น"}
]
# messages = [
#     {"role": "system", "content": "You are an expert Thai poet specializing in `กลอนแปด` (Eight-syllable verse) poetry. When a user provides a prompt, respond ONLY with a Thai poem (2-4 stanzas) that addresses their request, without any explanations or commentary. Your poem must strictly follow traditional กลอนแปด structure and rhyming patterns. Include phonetic rhyming tags for all rhyming words using the format: `<r>[vowel][ending consonant]word</r>` (examples: `<r>[a][w]เขา</r>`, `<r>[o][k]นก</r>`, `<r>[a]ผา</r>`, `<r>[i]ศรี</r>`). These tags should mark all external and internal rhymes according to proper กลอนแปด structure. Create vivid, culturally appropriate poetry that demonstrates mastery of Thai prosody while faithfully addressing the user's requested theme or scenario."},
#     {"role": "user", "content": "ช่วยแต่งกลอนที่เกี่ยวกับหนุ่มสาวเดินท่ามกลางสายฝน"}
# ]


input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
).to(device)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an AI specialized in Thai phonetic transcription. When given a Thai sentence in Thai script, your task is to output its phonetic transcription using the following format. Each syllable must be represented in a phoneme block as [StarterConsonant][Vowel][EndingConsonant][ToneNumber], omitting the EndingConsonant if it does not exist. Use the tilde symbol `~` to link syllables that are part of the same word, and use the vertical bar `|` to separate different words. For any short syllable, append a single apostrophe `'` immediately after its phoneme block. Do not include any glottal stop or aspiration markers in the transcription. Your output must contain only the transcription with no explanation, translation, or additional text.<|eot_id|><|start_header_id|>user<|end_header_id|>

และมันไม่ได้ดูดีอย่างที่เห็น<|eot_id|><|start_header_id|>assistant<|end_header_id|>

[l][x][3]|[m][a][n][0]|[m][a][j][2]~[d][aa][j][2]|[d][uu][

In [11]:
# messages = [
#     {"role": "system", "content": "You are an AI specialized in Thai phonetic transcription. When given a Thai sentence in Thai script, your task is to output its phonetic transcription using the following format. Each syllable must be represented in a phoneme block as [StarterConsonant][Vowel][EndingConsonant][ToneNumber], omitting the EndingConsonant if it does not exist. Use the tilde symbol `~` to link syllables that are part of the same word, and use the vertical bar `|` to separate different words. For any short syllable, append a single apostrophe `'` immediately after its phoneme block. Do not include any glottal stop or aspiration markers in the transcription. Your output must contain only the transcription with no explanation, translation, or additional text."},
#     {"role": "user", "content": "และมันไม่ได้ดูดีอย่างที่เห็น"}
# ]
messages = [
    {"role": "system", "content": "You are an expert Thai poet specializing in `กลอนแปด` (Eight-syllable verse) poetry. When a user provides a prompt, respond ONLY with a Thai poem (2-4 stanzas) that addresses their request, without any explanations or commentary. Your poem must strictly follow traditional กลอนแปด structure and rhyming patterns. Include phonetic rhyming tags for all rhyming words using the format: `<r>[vowel][ending consonant]word</r>` (examples: `<r>[a][w]เขา</r>`, `<r>[o][k]นก</r>`, `<r>[a]ผา</r>`, `<r>[i]ศรี</r>`). These tags should mark all external and internal rhymes according to proper กลอนแปด structure. Create vivid, culturally appropriate poetry that demonstrates mastery of Thai prosody while faithfully addressing the user's requested theme or scenario."},
    {"role": "user", "content": "ช่วยแต่งกลอนที่เกี่ยวกับหนุ่มสาวเดินท่ามกลางสายฝน"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
).to(device)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert Thai poet specializing in `กลอนแปด` (Eight-syllable verse) poetry. When a user provides a prompt, respond ONLY with a Thai poem (2-4 stanzas) that addresses their request, without any explanations or commentary. Your poem must strictly follow traditional กลอนแปด structure and rhyming patterns. Include phonetic rhyming tags for all rhyming words using the format: `<r>[vowel][ending consonant]word</r>` (examples: `<r>[a][w]เขา</r>`, `<r>[o][k]นก</r>`, `<r>[a]ผา</r>`, `<r>[i]ศรี</r>`). These tags should mark all external and internal rhymes according to proper กลอนแปด structure. Create vivid, culturally appropriate poetry that demonstrates mastery of Thai prosody while faithfully addressing the user's requested theme or scenario.<|eot_id|><|start_header_id|>user<|end_

In [6]:
messages = [
    {"role": "system", "content": "You are an AI specialized in Thai phonetic transcription. When given a Thai sentence in Thai script, your task is to output its phonetic transcription using the following format. Each syllable must be represented in a phoneme block as [StarterConsonant][Vowel][EndingConsonant][ToneNumber], omitting the EndingConsonant if it does not exist. Use the tilde symbol `~` to link syllables that are part of the same word, and use the vertical bar `|` to separate different words. For any short syllable, append a single apostrophe `'` immediately after its phoneme block. Do not include any glottal stop or aspiration markers in the transcription. Your output must contain only the transcription with no explanation, translation, or additional text."},
    {"role": "user", "content": "เข้ากันไม่ค่อยดี"}
]
# messages = [
#     {"role": "system", "content": "You are an expert Thai poet specializing in `กลอนแปด` (Eight-syllable verse) poetry. When a user provides a prompt, respond ONLY with a Thai poem (2-4 stanzas) that addresses their request, without any explanations or commentary. Your poem must strictly follow traditional กลอนแปด structure and rhyming patterns. Include phonetic rhyming tags for all rhyming words using the format: `<r>[vowel][ending consonant]word</r>` (examples: `<r>[a][w]เขา</r>`, `<r>[o][k]นก</r>`, `<r>[a]ผา</r>`, `<r>[i]ศรี</r>`). These tags should mark all external and internal rhymes according to proper กลอนแปด structure. Create vivid, culturally appropriate poetry that demonstrates mastery of Thai prosody while faithfully addressing the user's requested theme or scenario."},
#     {"role": "user", "content": "ช่วยแต่งกลอนที่เกี่ยวกับหนุ่มสาวเดินท่ามกลางสายฝน"}
# ]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
).to(device)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an AI specialized in Thai phonetic transcription. When given a Thai sentence in Thai script, your task is to output its phonetic transcription using the following format. Each syllable must be represented in a phoneme block as [StarterConsonant][Vowel][EndingConsonant][ToneNumber], omitting the EndingConsonant if it does not exist. Use the tilde symbol `~` to link syllables that are part of the same word, and use the vertical bar `|` to separate different words. For any short syllable, append a single apostrophe `'` immediately after its phoneme block. Do not include any glottal stop or aspiration markers in the transcription. Your output must contain only the transcription with no explanation, translation, or additional text.<|eot_id|><|start_header_id|>user<|end_header_id|>

เข้ากันไม่ค่อยดี<|eot_id|><|start_header_id|>assistant<|end_header_id|>

[kh][u]

In [ ]:
# messages = [
#     {"role": "system", "content": "You are an AI specialized in Thai phonetic transcription. When given a Thai sentence in Thai script, your task is to output its phonetic transcription using the following format. Each syllable must be represented in a phoneme block as [StarterConsonant][Vowel][EndingConsonant][ToneNumber], omitting the EndingConsonant if it does not exist. Use the tilde symbol `~` to link syllables that are part of the same word, and use the vertical bar `|` to separate different words. For any short syllable, append a single apostrophe `'` immediately after its phoneme block. Do not include any glottal stop or aspiration markers in the transcription. Your output must contain only the transcription with no explanation, translation, or additional text."},
#     {"role": "user", "content": "เข้ากันไม่ค่อยดี"}
# ]
messages = [
    {"role": "system", "content": "You are an expert Thai poet specializing in `กลอนแปด` (Eight-syllable verse) poetry. When a user provides a prompt, respond ONLY with a Thai poem (2-4 stanzas) that addresses their request, without any explanations or commentary. Your poem must strictly follow traditional กลอนแปด structure and rhyming patterns. Include phonetic rhyming tags for all rhyming words using the format: `<r>[vowel][ending consonant]word</r>` (examples: `<r>[a][w]เขา</r>`, `<r>[o][k]นก</r>`, `<r>[a]ผา</r>`, `<r>[i]ศรี</r>`). These tags should mark all external and internal rhymes according to proper กลอนแปด structure. Create vivid, culturally appropriate poetry that demonstrates mastery of Thai prosody while faithfully addressing the user's requested theme or scenario."},
    {"role": "user", "content": "ช่วยแต่งกลอนที่เกี่ยวกับหนุ่มสาวเดินท่ามกลางสายฝน"}
]



input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
).to(device)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

In [ ]:
# Placeholder for training command

git clone https://github.com/Pongsaky/LLaMA-Factory-klon8-generation.git
cd LLaMA-Factory-klon8-generation

ls
pip install -e .[torch,bitsandbytes,deepspeed,unsloth]
git checkout feat/poc-klon8-generation

wandb offline
export WANDB_MODE=offline

export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
FORCE_TORCHRUN=1 llamafactory-cli train train/llama3.2_instruct_full_sft.yaml

In [40]:
lora_result = get_data_from_json("eval_results/eval_results_llama3.2-typhoon2-1b-instruct-lora-klon8-nmt.json")
# full_result = get_data_from_json("eval_results/eval_results_llama3.2-typhoon2-1b-instruct-tagged_nmt-mixed.json")
full_result = get_data_from_json("eval_results/eval_results_llama3.2-typhoon2-1b-instruct-tagged_nmt-mixed copy.json")
len(lora_result), len(full_result)

(975, 983)

In [41]:
import pandas as pd

lora_df = pd.read_csv("eval_results/summary_eval_results_llama3.2-typhoon2-1b-instruct-lora-klon8-nmt.csv")
# full_df = pd.read_csv("eval_results/summary_eval_results_llama3.2-typhoon2-1b-instruct-tagged_nmt-mixed.csv")
full_df = pd.read_csv("eval_results/summary_eval_results-llama3.2-typhoon2-1b-instruct-tagged-nmt-mixed.csv")

len(lora_df), len(full_df)

(21, 21)

In [42]:
sucess_rhytm_row_index = (3, 12)
duplicate_row_index = (12, 21)

In [43]:
lora_df["0"][sucess_rhytm_row_index[0]:sucess_rhytm_row_index[1]].sum()

7360

In [44]:
# Success Rhyming
lora_success_rhytm_sum = lora_df["0"][sucess_rhytm_row_index[0]:sucess_rhytm_row_index[1]].sum()
full_success_rhytm_sum = full_df["0"][sucess_rhytm_row_index[0]:sucess_rhytm_row_index[1]].sum()

# Duplicate Rhyming
lora_duplicate_rhytm_sum = lora_df["0"][duplicate_row_index[0]:duplicate_row_index[1]].sum()
full_duplicate_rhytm_sum = full_df["0"][duplicate_row_index[0]:duplicate_row_index[1]].sum()

# Success Rhyming
lora_success_rhytm_sum, full_success_rhytm_sum, lora_duplicate_rhytm_sum, full_duplicate_rhytm_sum

(7360, 7607, 217, 245)

In [45]:
9*len(lora_result)

8775

In [46]:
lora_rhytm_accuracy = lora_success_rhytm_sum / (9*len(lora_result))
lora_dupliacte_rate = lora_duplicate_rhytm_sum / (9*len(lora_result))

full_rhytm_accuracy = full_success_rhytm_sum / (9*len(full_result))
full_dupliacte_rate = full_duplicate_rhytm_sum / (9*len(full_result))

lora_rhytm_accuracy, lora_dupliacte_rate, full_rhytm_accuracy, full_dupliacte_rate

(0.8387464387464387,
 0.02472934472934473,
 0.8598394936136543,
 0.027693003277947326)

In [1]:
0.8387464387464387 - 0.02472934472934473,  0.8598394936136543 - 0.027693003277947326

(0.814017094017094, 0.832146490335707)

### Megre LoRA to base model

In [1]:
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
import torch

device = "mps" if torch.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

model_id = "Pongsaky/llama3.2-typhoon2-1b-instruct-lora-klon8-nmt"
# model_id = "Pongsaky/llama3.2-typhoon2-1b-instruct-tagged_nmt-mixed"
model = AutoPeftModelForCausalLM.from_pretrained(model_id, tie_word_embeddings=False).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model.vocab_size, len(tokenizer)

[2025-05-19 10:40:51,095] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to mps (auto detect)


W0519 10:40:51.287000 60867 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of LlamaForCausalLM were not initialized from the model checkpoint at Pongsaky/llama3.2-typhoon2-1b-instruct-klon8-llama_factory and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(128334, 128334)

In [4]:
for name,param in model.named_parameters():
    print(name, param.shape)


base_model.model.model.embed_tokens.base_layer.weight torch.Size([128334, 2048])
base_model.model.model.embed_tokens.lora_embedding_A.default torch.Size([64, 128334])
base_model.model.model.embed_tokens.lora_embedding_B.default torch.Size([2048, 64])
base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight torch.Size([2048, 2048])
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([64, 2048])
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([2048, 64])
base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight torch.Size([512, 2048])
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.Size([64, 2048])
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.Size([512, 64])
base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight torch.Size([512, 2048])
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([64, 2048])
base_m

In [4]:
merged_model = model.merge_and_unload(progressbar=True, safe_merge=True)

Unloading and merging model:   0%|          | 0/329 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Unloading and merging model: 100%|██████████| 329/329 [00:08<00:00, 40.90it/s] 


In [5]:
# messages = [
#     {"role": "system", "content": "You are an AI specialized in Thai phonetic transcription. When given a Thai sentence in Thai script, your task is to output its phonetic transcription using the following format. Each syllable must be represented in a phoneme block as [StarterConsonant][Vowel][EndingConsonant][ToneNumber], omitting the EndingConsonant if it does not exist. Use the tilde symbol `~` to link syllables that are part of the same word, and use the vertical bar `|` to separate different words. For any short syllable, append a single apostrophe `'` immediately after its phoneme block. Do not include any glottal stop or aspiration markers in the transcription. Your output must contain only the transcription with no explanation, translation, or additional text."},
#     {"role": "user", "content": "เข้ากันไม่ค่อยดี"}
# ]
messages = [
    {"role": "system", "content": "You are an expert Thai poet specializing in `กลอนแปด` (Eight-syllable verse) poetry. When a user provides a prompt, respond ONLY with a Thai poem (2-4 stanzas) that addresses their request, without any explanations or commentary. Your poem must strictly follow traditional กลอนแปด structure and rhyming patterns. Include phonetic rhyming tags for all rhyming words using the format: `<r>[vowel][ending consonant]word</r>` (examples: `<r>[a][w]เขา</r>`, `<r>[o][k]นก</r>`, `<r>[a]ผา</r>`, `<r>[i]ศรี</r>`). These tags should mark all external and internal rhymes according to proper กลอนแปด structure. Create vivid, culturally appropriate poetry that demonstrates mastery of Thai prosody while faithfully addressing the user's requested theme or scenario."},
    {"role": "user", "content": "ช่วยแต่งกลอนที่เกี่ยวกับหนุ่มสาวเดินท่ามกลางสายฝน"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
).to(device)

attention_mask = (input_ids != tokenizer.pad_token_id).long().to(model.device)

In [8]:
outputs = merged_model.generate(
                input_ids,
                max_new_tokens=128,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.001,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an expert Thai poet specializing in `กลอนแปด` (Eight-syllable verse) poetry. When a user provides a prompt, respond ONLY with a Thai poem (2-4 stanzas) that addresses their request, without any explanations or commentary. Your poem must strictly follow traditional กลอนแปด structure and rhyming patterns. Include phonetic rhyming tags for all rhyming words using the format: `<r>[vowel][ending consonant]word</r>` (examples: `<r>[a][w]เขา</r>`, `<r>[o][k]นก</r>`, `<r>[a]ผา</r>`, `<r>[i]ศรี</r>`). These tags should mark all external and internal rhymes according to proper กลอนแปด structure. Create vivid, culturally appropriate poetry that demonstrates mastery of Thai prosody while faithfully addressing the user's requested theme or scenario.<|eot_id|><|start_header_id|>user<|end_header_id|>

ช่วยแต่งกลอนที่เกี่ยวกับหนุ่มสาวเดินท่ามกลางสายฝน<|eot_id|><|start_hea

In [9]:
outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=128,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.001,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an expert Thai poet specializing in `กลอนแปด` (Eight-syllable verse) poetry. When a user provides a prompt, respond ONLY with a Thai poem (2-4 stanzas) that addresses their request, without any explanations or commentary. Your poem must strictly follow traditional กลอนแปด structure and rhyming patterns. Include phonetic rhyming tags for all rhyming words using the format: `<r>[vowel][ending consonant]word</r>` (examples: `<r>[a][w]เขา</r>`, `<r>[o][k]นก</r>`, `<r>[a]ผา</r>`, `<r>[i]ศรี</r>`). These tags should mark all external and internal rhymes according to proper กลอนแปด structure. Create vivid, culturally appropriate poetry that demonstrates mastery of Thai prosody while faithfully addressing the user's requested theme or scenario.<|eot_id|><|start_header_id|>user<|end_header_id|>

ช่วยแต่งกลอนที่เกี่ยวกับหนุ่มสาวเดินท่ามกลางสายฝน<|eot_id|><|start_hea

In [4]:
import json

with open("sample_df.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

In [6]:
data

[{'id': 'UuMpKInyXJ',
  'story': 'เกตุไชยถูกจับได้ว่าขโมยเงินสดจากตู้อัตโนมัติในธนาคารขณะทำงานในตำแหน่งพนักงานจ่ายเงิน',
  'answers': "[{'เกตุไชย': '[2]'}]"},
 {'id': 'iVAHhVeECX',
  'story': 'จารุวัลลภ์ใช้ปืนข่มขู่พนักงานธนาคารเพื่อชิงเงินสด ขณะที่จตุฬสหลอกลวงลูกค้าผ่านเว็บไซต์ปลอมเพื่อฉ้อโกง',
  'answers': "[{'จารุวัลลภ์': '[2,5]'}, {'จตุฬส': '[6]'}]"},
 {'id': 'pHlJkvJB9D',
  'story': 'กัญญสรใช้ปืนยิงข่มขู่พนักงานร้านทองเพื่อให้เปิดตู้เซฟ และลักทรัพย์ในร้าน ขณะที่จักรศิวัชชิงกระเป๋าเงินจากลูกค้า',
  'answers': "[{'กัญญสร': '[2,5]'}, {'จักรศิวัช': '[2]'}]"},
 {'id': 'LjKKxZu9Bk',
  'story': 'กรวรรษ์นั่งเล่นกับแมวที่บ้าน',
  'answers': "[{'กรวรรษ์': '[1]'}]"},
 {'id': '3PiS3ipjTu',
  'story': 'จรัสพลขโมยเงินบริจาคจากงานศพ ขณะที่เกศญาณีทำลายกล้องวงจรปิด',
  'answers': "[{'จรัสพล': '[2]'}, {'เกศญาณี': '[5]'}]"},
 {'id': '1evMXN9MzX',
  'story': 'กรชฎาใช้ปืนยิงกฤษณ์ดนัยจากระยะไกลในสวนสาธารณะจนเธอล้มลงและเสียชีวิต',
  'answers': "[{'กรชฎา': '[5,7]'}, {'กฤษณ์ดนัย': '[1]'}]"},
 {'id': 'b0LL